# NHL Player Value Dashboard

Analyzing player performance relative to salary (AAV) across three position groups:

- **Forwards** — Points per \$1M
- **Defensemen** — Shots Blocked per \$1M
- **Goalies** — Goals Saved per \$1M

Data sources: Natural Stat Trick (skaters/goalies), public salary data (AAV).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'sans-serif'

skaters = pd.read_csv('skaters.csv')
goalies = pd.read_csv('goalies.csv')
salaries = pd.read_csv('salaries.csv')

for df in [skaters, goalies, salaries]:
    df.columns = df.columns.str.strip()

# Use 'all' situation to avoid double-counting special teams
skaters = skaters[skaters['situation'] == 'all'].copy()
goalies = goalies[goalies['situation'] == 'all'].copy()

skaters_sal = pd.merge(skaters, salaries, on='name')
goalies_sal = pd.merge(goalies, salaries, on='name')

print(f'Skaters with salary data: {len(skaters_sal)}')
print(f'Goalies with salary data: {len(goalies_sal)}')

## Forwards — Points per $1M

In [ ]:
forwards = skaters_sal[skaters_sal['position'].isin(['C', 'L', 'R'])].copy()
forwards['points_per_million'] = forwards['I_F_points'] / (forwards['AAV'] / 1_000_000)

top_forwards = forwards.sort_values('points_per_million', ascending=False)
print('Top 10 Forwards by Points per $1M:')
print(top_forwards[['name', 'position', 'I_F_points', 'AAV', 'points_per_million']]
      .head(10).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

ax.scatter(forwards['AAV'] / 1_000_000, forwards['points_per_million'],
           alpha=0.7, edgecolors='steelblue', color='lightsteelblue', s=60)

# Label top 5
for _, row in top_forwards.head(5).iterrows():
    ax.annotate(row['name'], (row['AAV'] / 1_000_000, row['points_per_million']),
                textcoords='offset points', xytext=(5, 4), fontsize=8)

ax.set_xlabel('Salary AAV ($M)', fontsize=11)
ax.set_ylabel('Points per $1M', fontsize=11)
ax.set_title('Forward Value: Points per $1M', fontsize=13, fontweight='bold')
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('forwards_value.png', bbox_inches='tight')
plt.show()

## Defensemen — Shots Blocked per $1M

In [ ]:
defense = skaters_sal[skaters_sal['position'] == 'D'].copy()
defense['shots_blocked_per_million'] = defense['shotsBlockedByPlayer'] / (defense['AAV'] / 1_000_000)

top_defense = defense.sort_values('shots_blocked_per_million', ascending=False)
print('Top 10 Defensemen by Shots Blocked per $1M:')
print(top_defense[['name', 'shotsBlockedByPlayer', 'AAV', 'shots_blocked_per_million']]
      .head(10).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

ax.scatter(defense['AAV'] / 1_000_000, defense['shots_blocked_per_million'],
           alpha=0.7, edgecolors='darkgreen', color='lightgreen', s=60)

for _, row in top_defense.head(5).iterrows():
    ax.annotate(row['name'], (row['AAV'] / 1_000_000, row['shots_blocked_per_million']),
                textcoords='offset points', xytext=(5, 4), fontsize=8)

ax.set_xlabel('Salary AAV ($M)', fontsize=11)
ax.set_ylabel('Shots Blocked per $1M', fontsize=11)
ax.set_title('Defenseman Value: Shots Blocked per $1M', fontsize=13, fontweight='bold')
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('defense_value.png', bbox_inches='tight')
plt.show()

## Goalies — Goals Saved per $1M

Goals saved is calculated as **shots on goal faced minus goals allowed** (i.e. saves made).

In [ ]:
goalies_sal['goals_saved'] = goalies_sal['ongoal'] - goalies_sal['goals']
goalies_sal['goals_saved_per_million'] = goalies_sal['goals_saved'] / (goalies_sal['AAV'] / 1_000_000)

top_goalies = goalies_sal.sort_values('goals_saved_per_million', ascending=False)
print('Goalies by Goals Saved per $1M:')
print(top_goalies[['name', 'ongoal', 'goals', 'goals_saved', 'AAV', 'goals_saved_per_million']]
      .to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

ax.scatter(goalies_sal['AAV'] / 1_000_000, goalies_sal['goals_saved_per_million'],
           alpha=0.8, edgecolors='firebrick', color='lightsalmon', s=80)

for _, row in goalies_sal.iterrows():
    ax.annotate(row['name'], (row['AAV'] / 1_000_000, row['goals_saved_per_million']),
                textcoords='offset points', xytext=(5, 4), fontsize=9)

ax.set_xlabel('Salary AAV ($M)', fontsize=11)
ax.set_ylabel('Goals Saved per $1M', fontsize=11)
ax.set_title('Goalie Value: Goals Saved per $1M', fontsize=13, fontweight='bold')
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('goalies_value.png', bbox_inches='tight')
plt.show()

## Combined Dashboard

In [ ]:
fig = plt.figure(figsize=(18, 6))
gs = gridspec.GridSpec(1, 3, figure=fig, wspace=0.35)

# --- Forwards ---
ax1 = fig.add_subplot(gs[0])
ax1.scatter(forwards['AAV'] / 1_000_000, forwards['points_per_million'],
            alpha=0.7, edgecolors='steelblue', color='lightsteelblue', s=55)
for _, row in top_forwards.head(3).iterrows():
    ax1.annotate(row['name'].split()[-1],
                 (row['AAV'] / 1_000_000, row['points_per_million']),
                 textcoords='offset points', xytext=(4, 3), fontsize=7)
ax1.set_xlabel('Salary AAV ($M)', fontsize=10)
ax1.set_ylabel('Points per $1M', fontsize=10)
ax1.set_title('Forwards', fontsize=12, fontweight='bold')
ax1.grid(True, linestyle='--', alpha=0.4)

# --- Defense ---
ax2 = fig.add_subplot(gs[1])
ax2.scatter(defense['AAV'] / 1_000_000, defense['shots_blocked_per_million'],
            alpha=0.7, edgecolors='darkgreen', color='lightgreen', s=55)
for _, row in top_defense.head(3).iterrows():
    ax2.annotate(row['name'].split()[-1],
                 (row['AAV'] / 1_000_000, row['shots_blocked_per_million']),
                 textcoords='offset points', xytext=(4, 3), fontsize=7)
ax2.set_xlabel('Salary AAV ($M)', fontsize=10)
ax2.set_ylabel('Shots Blocked per $1M', fontsize=10)
ax2.set_title('Defensemen', fontsize=12, fontweight='bold')
ax2.grid(True, linestyle='--', alpha=0.4)

# --- Goalies ---
ax3 = fig.add_subplot(gs[2])
ax3.scatter(goalies_sal['AAV'] / 1_000_000, goalies_sal['goals_saved_per_million'],
            alpha=0.8, edgecolors='firebrick', color='lightsalmon', s=70)
for _, row in goalies_sal.iterrows():
    ax3.annotate(row['name'].split()[-1],
                 (row['AAV'] / 1_000_000, row['goals_saved_per_million']),
                 textcoords='offset points', xytext=(4, 3), fontsize=7)
ax3.set_xlabel('Salary AAV ($M)', fontsize=10)
ax3.set_ylabel('Goals Saved per $1M', fontsize=10)
ax3.set_title('Goalies', fontsize=12, fontweight='bold')
ax3.grid(True, linestyle='--', alpha=0.4)

fig.suptitle('NHL Player Value Dashboard — Performance per $1M AAV', fontsize=15, fontweight='bold', y=1.02)
plt.savefig('nhl_value_dashboard.png', bbox_inches='tight')
plt.show()
print('Dashboard saved as nhl_value_dashboard.png')